In [ ]:
import pandas as pd
import json

# ==== ĐỌC FILE ====
with open("config_single.json", "r", encoding="utf-8") as file:
    config = json.load(file)

OUTPUT_CSV = config.get("OUTPUT_CSV")
# OUTPUT_CSV = "loc_single.csv"
print("OUTPUT_CSV : ", OUTPUT_CSV)
df = pd.read_csv(OUTPUT_CSV)
print(f"Số dòng ban đầu: {len(df)}")

# ==== ÉP KIỂU SỐ ====
numeric_cols = ["sharpe", "fitness", "LOW_2Y_SHARPE_value"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ==== LỌC ====
filtered_df = df[
    (
        (df["sharpe"] >= 0.8) &
        (
            (df["LOW_2Y_SHARPE_value"] > 0) |
            (df["LOW_2Y_SHARPE_value"].isna())
        )
    )
    |
    (
        (df["sharpe"] <= -0.8) &
        (
            (df["LOW_2Y_SHARPE_value"] < 0) |
            (df["LOW_2Y_SHARPE_value"].isna())
        )
    )
].copy()

# ==== FLIP ALPHA ÂM ====
mask_negative = filtered_df["sharpe"] <= -0.8

filtered_df.loc[mask_negative, ["sharpe", "fitness", "LOW_2Y_SHARPE_value"]] = (
    filtered_df.loc[mask_negative, ["sharpe", "fitness", "LOW_2Y_SHARPE_value"]].abs()
)

filtered_df.loc[mask_negative, "regular"] = (
    "-" + filtered_df.loc[mask_negative, "regular"].astype(str)
)

# ==== CHỌN CỘT ====
output_df = filtered_df[
    ["id", "sharpe", "fitness", "LOW_2Y_SHARPE_value", "regular"]
]

# ==== SORT (QUAN TRỌNG: để giữ bản tốt nhất) ====
output_df = output_df.sort_values(by="sharpe", ascending=False)

# ==== REMOVE DUPLICATE THEO REGULAR ====
output_df = output_df.drop_duplicates(subset=["regular"], keep="first")

# ==== RESET INDEX ====
output_df = output_df.reset_index(drop=True)

# ==== LƯU ====
output_df.to_csv("loc_single.csv", index=False)

print(f"Done ✅ Số alpha sau lọc & dedup: {len(output_df)}")

OUTPUT_CSV :  loc_single.csv
Số dòng ban đầu: 1151
Done ✅ Số alpha sau lọc & dedup: 1149


In [10]:
import pandas as pd
import random
import os
# Tham số
n = 6  # <-- Bạn có thể thay đổi giá trị này
num_formulas = 50000

# Đọc dữ liệu
df = pd.read_csv("loc_single.csv")

# Chia theo sharpe
regular_ge_1 = df[df["sharpe"] >= 0.8]["regular"].tolist()
regular_lt_1 = df[df["sharpe"] < 0.8]["regular"].tolist()

def generate_combination(n):
    p = random.random()
    
    # Chọn theo xác suất
    if p <= 1:
        selected = random.sample(regular_ge_1, min(n, len(regular_ge_1)))
    # elif p < 0.36:
    #     selected = random.sample(regular_lt_1, min(n, len(regular_lt_1)))
    # else:
    #     max_ge = min(len(regular_ge_1), n - 2)
    #     count_ge = random.randint(2, max_ge) if max_ge >= 2 else max_ge
    #     count_lt = n - count_ge
    #     if count_ge > 0 and count_lt > 0:
    #         selected = random.sample(regular_ge_1, count_ge) + \
    #                    random.sample(regular_lt_1, min(count_lt, len(regular_lt_1)))
    #     else:
    #         selected = random.sample(regular_ge_1 + regular_lt_1, n)

    # Shuffle để đảm bảo thứ tự ngẫu nhiên
    random.shuffle(selected)

    # Gán dấu trừ ngẫu nhiên (nếu không phải xác suất 0.25)
    if random.random() >= 0.95:
        count_minus = random.randint(1, min(2, len(selected)))
        minus_indices = random.sample(range(len(selected)), count_minus)
        # print("minus_indices : ", minus_indices)
        
        for i in minus_indices:
            expr = selected[i].lstrip()
            if expr.startswith("-"):
                # Nếu đã có dấu '-', thêm nữa thành '--' → chuyển thành không có dấu
                selected[i] = expr.lstrip("-+ ")
            else:
                # Nếu chưa có '-', thêm vào
                selected[i] = "-" + expr.lstrip("+- ")
            # print("selected[i] : ", selected[i])

    # Ghép công thức với logic đúng dấu
    expression = selected[0].strip()
    for expr in selected[1:]:
        expr = expr.strip()
        if expr.startswith("-"):
            # Nếu biểu thức đã có dấu trừ → + -expr
            expression += " + " + expr
        elif expr.startswith("+"):
            expression += " + " + expr[1:]
        else:
            expression += " + " + expr

    # Thay thế các chuỗi '+ -' bằng '-'
    expression = expression.replace("+ -", "- ")

    return expression


# Tạo 10,000 công thức (cho phép trùng lặp nhẹ)
formulas = []
for _ in range(num_formulas):
    n = random.choices([6, 7, 8], weights=[0.3, 0.4, 0.3])[0]
    formula = generate_combination(n)
    formulas.append(formula)

# Ghi ra file
with open("alpha_single.txt", "w") as f:
    for formula in formulas:
        f.write(formula + "\n")

print(f"✅ Đã tạo {num_formulas} công thức với {n} regular mỗi công thức.")


✅ Đã tạo 50000 công thức với 7 regular mỗi công thức.
